# S5-extras — CQR localmente adaptativo + análises post-hoc (Kaggle)

**Autor**: Massanori
**Data**: 03/06/2026
**Descrição**: Orquestra os 5 itens da branch `feat/S5-extras` sem re-treino, reaproveitando
checkpoints (A/B/C), recons (cal/test) e os `metrics_*.csv` do S5.8.

- **Item 1** — CQR localmente adaptativo (isola o mecanismo do achado ResM)
- **Itens 2 + 4** — varredura de nível nominal → fronteira de eficiência + curva de confiabilidade
- **Item 3** — gap de cobertura condicional (Clopper-Pearson)
- **Item 5** — IC BCa + tamanho de efeito sobre o delta pareado

## Pré-requisitos (Settings → Add Input)
1. `tcc-mri-recons-varnet-brain-4x` (splits cal/test)
2. `tcc-mri-resm-checkpoints` (Grupo A)
3. `tcc-mri-qr-checkpoints` (Grupo B)
4. `tcc-mri-qr-lesion-checkpoints` (Grupo C)
5. `tcc-mri-lesion-masks` (máscaras)
6. **metrics do S5.8** — o dataset/arquivos com `metrics_A.csv`, `metrics_B.csv`, `metrics_C.csv`
   (itens 3 e 5). Se você não publicou como dataset, suba os 3 CSVs como um dataset novo
   ou rode os itens 3/5 localmente no VS Code.

**Accelerator**: `None` (CPU) preserva a cota T4 — como no S5.7. Pode trocar para T4 para acelerar.

In [ ]:
# Célula 2 — Setup: clone da branch feat/S5-extras + dependências
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/KR0N0S7/tcc-mri-uncertainty.git'
REPO_DIR = Path('/kaggle/working/tcc-mri-uncertainty')
BRANCH = 'feat/S5-extras'

if REPO_DIR.is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=False)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('HEAD:')
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO_DIR / 'requirements.txt'), 'python-dotenv'], check=True)
print('Setup OK')

In [ ]:
# Célula 3 — Diagnóstico: localiza recons, 3 checkpoints, máscaras e metrics_*.csv
# Autor: Massanori
# Data: 03/06/2026
# Descrição: valida device e resolve os caminhos de todos os inputs Kaggle,
# definindo variáveis globais (RECONS_ROOT, CHECKPOINT_PATHS, MASKS_ROOT,
# METRICS_DIR) para as células seguintes.
import torch

RECONS_SLUG = 'tcc-mri-recons-varnet-brain-4x'
MASKS_SLUG  = 'tcc-mri-lesion-masks'
CKPT_SLUGS  = {'A': 'tcc-mri-resm-checkpoints',
               'B': 'tcc-mri-qr-checkpoints',
               'C': 'tcc-mri-qr-lesion-checkpoints'}

def candidates(slug):
    cs = [Path('/kaggle/input') / slug]
    root = Path('/kaggle/input/datasets')
    if root.is_dir():
        for u in root.iterdir():
            if u.is_dir():
                cs.append(u / slug)
    return cs

def find_recons_root(slug):
    for c in candidates(slug):
        if c.is_dir():
            subs = {p.name for p in c.iterdir() if p.is_dir()}
            if {'cal', 'test'}.issubset(subs):
                return c
            sd = [p for p in c.iterdir() if p.is_dir()]
            if len(sd) == 1 and (sd[0] / 'cal').is_dir():
                return sd[0]
    raise FileNotFoundError(f'Recons "{slug}" sem cal/test.')

def find_dir_with(slug, marker_glob):
    for c in candidates(slug):
        if c.is_dir():
            if list(c.glob(marker_glob)):
                return c
            hits = [p.parent for p in c.rglob(marker_glob)
                    if '.ipynb_checkpoints' not in p.parts]
            if hits:
                return hits[0]
    raise FileNotFoundError(f'"{slug}" sem {marker_glob}.')

def find_checkpoint(slug):
    for c in candidates(slug):
        if c.is_dir():
            if (c / 'best.pt').is_file():
                return c / 'best.pt'
            hits = [p for p in c.rglob('best.pt') if '.ipynb_checkpoints' not in p.parts]
            if hits:
                return sorted(hits, key=lambda p: len(p.parts))[0]
    raise FileNotFoundError(f'"{slug}" sem best.pt.')

HAS_CUDA = torch.cuda.is_available()
DEVICE = 'cuda' if HAS_CUDA else 'cpu'
print(f'[device] {DEVICE}')

RECONS_ROOT = find_recons_root(RECONS_SLUG)
print(f'[recons] {RECONS_ROOT} | cal={len(list((RECONS_ROOT/"cal").glob("*.npz")))} '
      f'test={len(list((RECONS_ROOT/"test").glob("*.npz")))}')

MASKS_ROOT = find_dir_with(MASKS_SLUG, '*.pt')   # mascaras sao .pt (geradas na S3)
print(f'[masks]  {MASKS_ROOT} | {len(list(MASKS_ROOT.glob("*.pt")))} arquivos .pt')

CHECKPOINT_PATHS = {g: find_checkpoint(s) for g, s in CKPT_SLUGS.items()}
for g, p in CHECKPOINT_PATHS.items():
    print(f'[ckpt {g}] {p} ({p.stat().st_size/1e6:.1f} MB)')

# metrics_*.csv (itens 3 e 5): procura em qualquer input
METRICS_DIR = None
for hit in Path('/kaggle/input').rglob('metrics_A.csv'):
    if '.ipynb_checkpoints' not in hit.parts:
        METRICS_DIR = hit.parent
        break
print(f'[metrics] {METRICS_DIR if METRICS_DIR else "NAO ENCONTRADO — itens 3/5 serão pulados"}')

os.environ['TCC_RECONS_DIR'] = str(RECONS_ROOT)
os.environ['TCC_ANOTADOS_DIR'] = '/kaggle/working/_dummy'
os.environ['TCC_BRAIN_CSV'] = '/kaggle/working/_dummy.csv'
Path('/kaggle/working/_dummy').mkdir(exist_ok=True); Path('/kaggle/working/_dummy.csv').touch()
OUT = Path('/kaggle/working/results'); OUT.mkdir(exist_ok=True)
print('Diagnóstico OK')

In [ ]:
# Célula 4 — Itens 3 e 5 (CPU, segundos) sobre os metrics_*.csv do S5.8
# Item 3: gap de cobertura condicional (Clopper-Pearson)
# Item 5: IC BCa + rank-biserial + d_z sobre o delta pareado
if METRICS_DIR is None:
    print('metrics_*.csv não encontrado — pule para a Célula 5 (sweep) e rode 3/5 localmente.')
else:
    subprocess.run([sys.executable, 'scripts/analyze_conditional_coverage.py',
                    '--metrics-dir', str(METRICS_DIR),
                    '--output', str(OUT / 'cond_cov.csv')], check=True)
    print('\n' + '=' * 60 + '\n')
    subprocess.run([sys.executable, 'scripts/analyze_paired_deltas.py',
                    '--metrics-dir', str(METRICS_DIR),
                    '--output', str(OUT / 'paired.csv')], check=True)

In [ ]:
# Célula 5 — Itens 1+2+4: varredura de nível nominal (5 calibradores)
# A/scaled, B/cqr, B/cqr_norm, C/cqr, C/cqr_norm. Cada um é idempotente
# (grava seu CSV); se a sessão cair, reexecute só os que faltam.
RUNS = [
    ('A', 'scaled',   CHECKPOINT_PATHS['A']),
    ('B', 'cqr',      CHECKPOINT_PATHS['B']),
    ('B', 'cqr_norm', CHECKPOINT_PATHS['B']),
    ('C', 'cqr',      CHECKPOINT_PATHS['C']),
    ('C', 'cqr_norm', CHECKPOINT_PATHS['C']),
]
for group, calib, ckpt in RUNS:
    out_csv = OUT / f'sweep_{group}_{calib}.csv'
    if out_csv.is_file():
        print(f'[skip] {out_csv.name} já existe')
        continue
    print(f'\n=== sweep {group}/{calib} ===')
    r = subprocess.run([sys.executable, 'scripts/analyze_calibration_sweep.py',
                        '--group', group, '--calibrator', calib,
                        '--checkpoint', str(ckpt),
                        '--recons-dir', str(RECONS_ROOT),
                        '--masks-dir', str(MASKS_ROOT),
                        '--output', str(out_csv),
                        '--device', DEVICE])
    assert r.returncode == 0, f'sweep {group}/{calib} falhou (rc={r.returncode})'
print('\nVarredura concluída.')

In [ ]:
# Célula 6 — Itens 2+4: figuras (reliability + efficiency) + tabela 0.90
subprocess.run([sys.executable, 'scripts/plot_calibration_extras.py',
                '--sweep-dir', str(OUT), '--out-dir', str(OUT)], check=True)

from IPython.display import Image, display
for png in ['reliability_curve.png', 'efficiency_frontier.png']:
    p = OUT / png
    if p.is_file():
        print(png); display(Image(str(p)))

In [ ]:
# Célula 7 — (opcional) validação cruzada: o nível 0.90 do sweep deve bater
# com os números do S5.8 (coverage_lesion A~0.869, B~0.784, C~0.786).
import pandas as pd, glob
rows = [pd.read_csv(f) for f in glob.glob(str(OUT / 'sweep_*.csv'))]
if rows:
    df = pd.concat(rows, ignore_index=True)
    s = df[abs(df['nominal_coverage'] - 0.90) < 1e-9]
    cols = ['group','calibrator','q_hat','coverage_global','coverage_lesion',
            'width_global','width_lesion']
    with pd.option_context('display.float_format', lambda x: f'{x:.4f}'):
        print(s[cols].sort_values(['group','calibrator']).to_string(index=False))

In [ ]:
# Célula 8 — Mondrian: calibração por sequência (marginal vs mondrian)
# Endereça o achado do item 3 (sub-cobertura em AXT1). Mesmos inputs do sweep
# (recons cal/test + checkpoints + máscaras); NÃO usa os metrics_*.csv do S5.8.
RUNS_MONDRIAN = [
    ('A', 'scaled',   CHECKPOINT_PATHS['A']),
    ('B', 'cqr',      CHECKPOINT_PATHS['B']),
    ('C', 'cqr',      CHECKPOINT_PATHS['C']),
    ('B', 'cqr_norm', CHECKPOINT_PATHS['B']),
    ('C', 'cqr_norm', CHECKPOINT_PATHS['C']),
]
for group, calib, ckpt in RUNS_MONDRIAN:
    out_csv = OUT / f'mondrian_{group}_{calib}.csv'
    if out_csv.is_file():
        print(f'[skip] {out_csv.name} já existe'); continue
    print(f'\n=== mondrian {group}/{calib} ===', flush=True)
    r = subprocess.run([sys.executable, 'scripts/analyze_mondrian_coverage.py',
                        '--group', group, '--calibrator', calib,
                        '--checkpoint', str(ckpt),
                        '--recons-dir', str(RECONS_ROOT),
                        '--masks-dir', str(MASKS_ROOT),
                        '--output', str(out_csv),
                        '--device', DEVICE],
                       capture_output=True, text=True)
    # o resumo "COBERTURA EM LESAO por sequencia" sai no stderr do script:
    if r.stdout: print(r.stdout[-1500:])
    if r.stderr: print(r.stderr[-1500:])
    assert r.returncode == 0, f'mondrian {group}/{calib} falhou (rc={r.returncode})'
print('\nMondrian concluído. CSVs em', OUT)

In [ ]:
# Célula 9 — Mondrian: tabela marginal vs mondrian (cobertura em lesão por sequência)
import pandas as pd, glob
print('Conteúdo de', OUT, ':')
for p in sorted(OUT.glob('*')):
    print('  ', p.name)

files = sorted(glob.glob(str(OUT / 'mondrian_*.csv')))
if not files:
    print('\nNenhum mondrian_*.csv — rode a Célula 8 antes.')
for f in files:
    df = pd.read_csv(f)
    g, c = df['group'].iloc[0], df['calibrator'].iloc[0]
    print(f'\n=== {g} / {c} : cobertura em lesão (alvo 0.90) ===')
    piv = (df[df['stratum'] != 'ALL']
           .pivot_table(index='stratum', columns='scheme', values='coverage_lesion'))
    if 'mondrian' in piv and 'marginal' in piv:
        piv['delta (mond-marg)'] = piv['mondrian'] - piv['marginal']
    with pd.option_context('display.float_format', lambda x: f'{x:.4f}'):
        print(piv.to_string())

## Saídas (em `/kaggle/working/results/`)
- `cond_cov.csv` (item 3), `paired.csv` (item 5)
- `sweep_{A_scaled,B_cqr,B_cqrnorm,C_cqr,C_cqrnorm}.csv` (itens 1/2/4)
- `mondrian_{A_scaled,B_cqr,C_cqr,B_cqrnorm,C_cqrnorm}.csv` (Mondrian por sequência)
- `reliability_curve.png`, `efficiency_frontier.png`, `summary_level090.csv`

## Próximos passos
1. Baixe a pasta `results/` (Output do notebook) ou publique como dataset `tcc-mri-s5-extras`.
2. Confira a leitura do **item 1** na `summary_level090.csv`: se `cqr_norm` recupera a
   `coverage_lesion` próxima do `scaled` (ResM), o ganho vem da adaptatividade local da calibração.
3. Mondrian (Células 8-9): comparar `coverage_lesion` em AXT1 entre marginal e mondrian.
4. Com os números reais em mãos, peça a atualização do `docs/S5_extras.md`.